In [1]:
path = "/home/yamane/transformerCPI/data/classAByProtein/"

In [2]:
import pandas as pd

file_path = "/data/yamane/GPCRclassAData/"
fle_name = ["chemblSmiles.csv","classA_ligand_binary_202110.csv","id2seq.csv"]

smiles_df = pd.read_csv(file_path+fle_name[0]).drop("Unnamed: 0",axis = 1)
origin_df = pd.read_csv("origin.csv").drop("Unnamed: 0",axis = 1)
seq_df = pd.read_csv(file_path+fle_name[2]).drop("Unnamed: 0",axis = 1)

print("interaction : ",len(origin_df))
print("protein_num : ",len(set(origin_df["UniProt ID"].to_list())))
print("ligand_num : ",len(set(origin_df["Database Ligand ID"].to_list())))

interaction :  337115
protein_num :  504
ligand_num :  143807


In [3]:
s = 0
test_index = []
for i,v in zip(list(origin_df["UniProt ID"].value_counts().index),list(origin_df["UniProt ID"].value_counts().values)):
  if v > 1000 and v < 1600:
    s += v
    test_index.append(i)
    #print(i,v)
print(s)

#testdata daraframe
for i,index in enumerate(test_index):
  if i == 0:
    test_df = origin_df[origin_df["UniProt ID"] == index]
  else:
    test_df = pd.concat([test_df,origin_df[origin_df["UniProt ID"] == index]])

test_df = test_df.reset_index().drop("index",axis = 1)
test_df.to_csv(path+"byProtein_test.csv")


49535


In [51]:
for i,name in enumerate(test_index):
  if i == 0:
    train_df = origin_df[origin_df["UniProt ID"] != name]
  else:
    train_df = train_df[train_df["UniProt ID"] != name]

train_df = train_df.reset_index().drop("index",axis = 1)
train_df.to_csv(path+"byProtein_train.csv")

In [52]:
def change_format(df):
  tcpi_format = []
  for i in range(len(df)):
    cid = df.iloc[i]["Database Ligand ID"]
    smiles = smiles_df[smiles_df["CHEMBL_ID"] == cid]["smiles"].values[0]
    pid = df.iloc[i]["UniProt ID"]
    seq = seq_df[seq_df["UniProt ID"] == pid]["sequence"].values[0]
    it = df.iloc[i]["Interaction_type"]
    tcpi_format.append(" ".join([smiles,seq,str(it)]))
  return tcpi_format

In [53]:
tcpi_format_train = change_format(train_df)

In [60]:
tcpi_format_train[1]

'C[C@@H]1NC(=O)N(C)[C@H](O)NC(=O)[C@@H](Cc2c[nH]c3ccccc23)NC(=O)[C@H](Cc2ccccc2)NC(=O)[C@@H](Cc2ccccc2)NC(=O)[C@H](N)CSSC[C@H](C(=O)O)NC(=O)[C@H](Cc2ccccc2)NC1=O MDMLHPSSVSTTSEPENASSAWPPDATLGNVSAGPSPAGLAVSGVLIPLVYLVVCVVGLLGNSLVIYVVLRHTASPSVTNVYILNLALADELFMLGLPFLAAQNALSYWPFGSLMCRLVMAVDGINQFTSIFCLTVMSVDRYLAVVHPTRSARWRTAPVARTVSAAVWVASAVVVLPVVVFSGVPRGMSTCHMQWPEPAAAWRAGFIIYTAALGFFGPLLVICLCYLLIVVKVRSAGRRVWAPSCQRRRRSERRVTRMVVAVVALFVLCWMPFYVLNIVNVVCPLPEEPAFFGLYFLVVALPYANSCANPILYGFLSYRFKQGFRRVLLRPSRRVRSQEPTVGPPEKTEEEDEEEEDGEESREGGKGKEMNGRVSQITQPGTSGQERPPSRVASKEQQLLPQEASTGEKSSTMRISYL 0'

In [54]:
tcpi_format_test = change_format(test_df)

In [63]:
with open(path+"byProtein_train.txt","w") as f:
  f.write('\n'.join(tcpi_format_train))